In [1]:
!pip install -q transformers sentencepiece accelerate sacremoses tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 24.2 MB/s eta 0:00:00


In [2]:
import json
import re
import zipfile
from pathlib import Path
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CUDA available: True
GPU: Tesla T4


In [3]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

ZIP_PATH = "/content/to_translate.zip"
EXTRACT_DIR = "/content/to_translate"

# push GPU harder
BATCH_SIZE = 128     # increase to 64 if VRAM allows

MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 128

LANG_MAP = {
    "hindi": "hin_Deva",
    "tamil": "tam_Taml",
    "telugu": "tel_Telu",
    "kannada": "kan_Knda",
    "malayalam": "mal_Mlym",
    "marathi": "mar_Deva",
    "gujarati": "guj_Gujr",
    "bengali": "ben_Beng"
}

TIMESTAMP_RE = re.compile(r"^\[(\d\d:\d\d:\d\d\.\d\d\d)\]\s*(.*)$")


In [4]:
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

model.eval()

print("Model ready.")


Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model ready.


In [5]:
print("Unzipping...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

root = Path(EXTRACT_DIR)

assert root.exists(), "Extraction failed!"

json_files = sorted(root.rglob("*.json"))

print("JSON files found:", len(json_files))


Unzipping...
JSON files found: 224


In [6]:
def split_timestamp(line):
    """
    '[00:00:01.230] hello' -> ('00:00:01.230', 'hello')
    """
    m = TIMESTAMP_RE.match(line)
    if not m:
        return None, line
    return m.group(1), m.group(2)


def translate_sentences_batched(sentences, src_lang):
    """
    GPU-batched translation of individual captions.
    """

    tokenizer.src_lang = src_lang
    forced_id = tokenizer.convert_tokens_to_ids("eng_Latn")

    outputs = []

    total_batches = (len(sentences) - 1) // BATCH_SIZE + 1

    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS
        ).to(device)

        with torch.no_grad():
            tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_id,
                max_new_tokens=MAX_NEW_TOKENS
            )

        decoded = tokenizer.batch_decode(
            tokens,
            skip_special_tokens=True
        )

        outputs.extend(decoded)

    return outputs


In [7]:
def process_file(path: Path):
    data = json.loads(path.read_text(encoding="utf-8"))

    # locate transcription key
    src_key = None
    for k in data:
        if k.startswith("transcription_") and k != "transcription_english":
            src_key = k
            break

    if not src_key:
        raise RuntimeError("No transcription_* key")

    lang_name = src_key.replace("transcription_", "").lower()

    if lang_name not in LANG_MAP:
        raise RuntimeError(f"Unsupported language: {lang_name}")

    src_lang_code = LANG_MAP[lang_name]

    lines = data[src_key]

    timestamps = []
    texts = []

    for l in lines:
        ts, txt = split_timestamp(l)
        timestamps.append(ts)
        texts.append(txt)

    # translate
    translated = translate_sentences_batched(texts, src_lang_code)

    if len(translated) != len(texts):
        raise RuntimeError("Caption count mismatch")

    rebuilt = []
    for ts, txt in zip(timestamps, translated):
        rebuilt.append(f"[{ts}] {txt}" if ts else txt)

    data["transcription_english"] = rebuilt

    path.write_text(
        json.dumps(data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )


In [ ]:
processed = set()
failed = []

print("Starting batch translation...")

for path in tqdm(json_files):

    if path in processed:
        continue

    try:
        process_file(path)
        processed.add(path)

    except Exception as e:
        print("\nFAILED:", path)
        print(e)
        failed.append(path)

print("\nDONE.")
print("Processed:", len(processed))
print("Failed:", len(failed))

if failed:
    print("\nFailed files:")
    for f in failed:
        print(f)


Starting batch translation...


  8%|▊         | 19/224 [12:49<2:50:55, 50.03s/it]

RETRY

In [1]:
!pip install -q transformers sentencepiece accelerate sacremoses tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 13.7 MB/s eta 0:00:00


In [2]:
import json
import re
import zipfile
from pathlib import Path
import torch
from tqdm import tqdm

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CUDA available: True
GPU: Tesla T4


In [3]:
MODEL_NAME = "facebook/nllb-200-distilled-600M"

ZIP_PATH = "/content/to_translate.zip"
EXTRACT_DIR = "/content/to_translate"

# multi-file window size
FILES_PER_GROUP = 5

# GPU batching
BATCH_SIZE = 192        # crank it up
MAX_INPUT_TOKENS = 256
MAX_NEW_TOKENS = 128

LANG_MAP = {
    "hindi": "hin_Deva",
    "tamil": "tam_Taml",
    "telugu": "tel_Telu",
    "kannada": "kan_Knda",
    "malayalam": "mal_Mlym",
    "marathi": "mar_Deva",
    "gujarati": "guj_Gujr",
    "bengali": "ben_Beng"
}

TIMESTAMP_RE = re.compile(r"^\[(\d\d:\d\d:\d\d\.\d\d\d)\]\s*(.*)$")


In [4]:
print("Loading model...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

model.eval()

print("Model ready.")


Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model ready.


In [5]:
print("Extracting ZIP...")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall("/content")

root = Path(EXTRACT_DIR)
assert root.exists(), "Extraction failed!"

json_files = sorted(root.rglob("*.json"))

print("Total JSON files:", len(json_files))


Extracting ZIP...
Total JSON files: 224


In [6]:
def split_timestamp(line):
    m = TIMESTAMP_RE.match(line)
    if not m:
        return None, line
    return m.group(1), m.group(2)


def batched_translate(sentences, src_lang):
    """
    Translate a flat list of sentences using GPU batching.
    Order is preserved.
    """

    tokenizer.src_lang = src_lang
    forced_id = tokenizer.convert_tokens_to_ids("eng_Latn")

    outputs = []

    total_batches = (len(sentences) - 1) // BATCH_SIZE + 1

    for i in range(0, len(sentences), BATCH_SIZE):
        batch = sentences[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_TOKENS
        ).to(device)

        with torch.no_grad():
            tokens = model.generate(
                **inputs,
                forced_bos_token_id=forced_id,
                max_new_tokens=MAX_NEW_TOKENS
            )

        decoded = tokenizer.batch_decode(
            tokens,
            skip_special_tokens=True
        )

        outputs.extend(decoded)

    return outputs


In [7]:
def group_list(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i + n]


In [8]:
processed = set()
failed = []

print("Starting high-throughput translation...")

for group in tqdm(list(group_list(json_files, FILES_PER_GROUP))):

    # -----
    # STEP 1: read all files in this window
    # -----
    # print(" -> Reading all files in present window")
    file_data = {}
    caption_maps = {}
    flat_sentences = []
    flat_index = []

    for path in group:

        if path in processed:
            continue

        try:
            data = json.loads(path.read_text(encoding="utf-8"))

            # locate transcription key
            src_key = None
            for k in data:
                if k.startswith("transcription_") and k != "transcription_english":
                    src_key = k
                    break

            if not src_key:
                raise RuntimeError("No transcription_* key")

            lang_name = src_key.replace("transcription_", "").lower()

            if lang_name not in LANG_MAP:
                raise RuntimeError(f"Unsupported language: {lang_name}")

            lines = data[src_key]

            timestamps = []
            texts = []

            for l in lines:
                ts, txt = split_timestamp(l)
                timestamps.append(ts)
                texts.append(txt)

            file_data[path] = {
                "json": data,
                "src_key": src_key,
                "lang": lang_name,
                "timestamps": timestamps,
                "texts": texts,
            }

            # flatten
            for i, t in enumerate(texts):
                flat_sentences.append(t)
                flat_index.append((path, i))

        except Exception as e:
            print("\nFAILED loading:", path)
            print(e)
            failed.append(path)

    if not flat_sentences:
        continue

    # -----
    # STEP 2: translate giant batch
    # -----
    # print(" -> Batch translating all files")
    # NOTE: we group by language to avoid mixing src langs
    by_lang = {}
    for idx, (path, line_idx) in enumerate(flat_index):
        lang = file_data[path]["lang"]
        by_lang.setdefault(lang, []).append(idx)

    translated_global = [None] * len(flat_sentences)

    for lang, indices in by_lang.items():

        src_lang_code = LANG_MAP[lang]

        sentences = [flat_sentences[i] for i in indices]

        translated = batched_translate(sentences, src_lang_code)

        if len(translated) != len(indices):
            raise RuntimeError("Batch mismatch")

        for j, global_idx in enumerate(indices):
            translated_global[global_idx] = translated[j]

    # -----
    # STEP 3: route translations back
    # -----
    for (path, idx), text in zip(flat_index, translated_global):

        if text is None:
            raise RuntimeError("Missing translation")

        file_data[path].setdefault("translated", {})[idx] = text

    # -----
    # STEP 4: rebuild + save each file
    # -----
    # print(" -> Saving back translations to files")
    for path, info in file_data.items():

        texts = info["texts"]
        timestamps = info["timestamps"]

        rebuilt = []
        for i in range(len(texts)):
            rebuilt.append(
                f"[{timestamps[i]}] {info['translated'][i]}"
                if timestamps[i]
                else info['translated'][i]
            )

        info["json"]["transcription_english"] = rebuilt

        path.write_text(
            json.dumps(info["json"], ensure_ascii=False, indent=2),
            encoding="utf-8"
        )

        processed.add(path)

print("\nDONE.")
print("Processed:", len(processed))
print("Failed:", len(failed))


Starting high-throughput translation...


100%|██████████| 45/45 [1:47:26<00:00, 143.25s/it]


DONE.
Processed: 224
Failed: 0


In [9]:
import shutil
from pathlib import Path

SOURCE_DIR = "/content/to_translate"
ZIP_OUT = "/content/translated.zip"

# remove old zip if exists
zip_path = Path(ZIP_OUT)
if zip_path.exists():
    zip_path.unlink()

# create zip
shutil.make_archive(
    base_name=ZIP_OUT.replace(".zip", ""),
    format="zip",
    root_dir=SOURCE_DIR
)

print("Created:", ZIP_OUT)

# download in Colab
from google.colab import files
files.download(ZIP_OUT)


Created: /content/translated.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>